# 🎬 ROTBOTS — Archive.org Scraper

---

Download and segment videos from the Internet Archive into reusable clips.

These clips become **found footage** that can be mixed with AI-generated video in your final output.

> **🤔** What happens when you mix "real" archival footage with AI-generated video? Can you tell the difference?

---

In [ ]:
# SETUP
!pip install -q yt-dlp
import json, re, subprocess, shutil
from pathlib import Path
from IPython.display import display, Markdown, Video

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
TEMP = Path('/content/temp_archive'); TEMP.mkdir(exist_ok=True)
print('✅ Setup complete')

In [ ]:
# SELECT SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'session_info.json').exists()])
if not sessions: print('❌ No sessions! Run 02_Script_Writer first.')
else:
    print('📂 Sessions:')
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
VIDEOS_DIR = SESSION_DIR / 'videos'; VIDEOS_DIR.mkdir(exist_ok=True)
ARCHIVE_DIR = SESSION_DIR / 'archive_clips'; ARCHIVE_DIR.mkdir(exist_ok=True)
print(f'✅ Session: {SESSION_NAME}')

---
## 📥 Enter Archive.org URLs

Paste one or more Internet Archive video URLs below.

Supported formats:
- `https://archive.org/details/IDENTIFIER`
- `https://archive.org/download/IDENTIFIER/file.mp4`
- Just the identifier: `IDENTIFIER`

The system will download the video, split it into clips, and save them to your session.

In [ ]:
# ============================================================
# ARCHIVE URLs — paste your links here
# ============================================================

ARCHIVE_URLS = [
    # 'https://archive.org/details/example_video',
    # 'another_archive_id',
]

# Settings
PREFER_HEIGHT = 480       # Download quality (360, 480, 720)
MAX_CLIP_DURATION = 10    # Max seconds per clip (shorter = more clips)
MIN_CLIP_DURATION = 3     # Skip clips shorter than this

print(f'🎬 {len(ARCHIVE_URLS)} archive URL(s)')
print(f'   Quality: {PREFER_HEIGHT}p, clips: {MIN_CLIP_DURATION}-{MAX_CLIP_DURATION}s')

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_archive_id(url):
    url = url.strip().rstrip('/')
    m = re.search(r'archive\.org/(?:details|download)/([^/?#]+)', url)
    if m: return m.group(1)
    if '/' not in url and '.' not in url: return url
    raise ValueError(f'Cannot parse archive ID from: {url}')

def get_dur(path):
    try:
        r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration',
            '-of','default=noprint_wrappers=1:nokey=1',str(path)], capture_output=True, text=True, timeout=10)
        return float(r.stdout.strip())
    except: return 0

def download_archive_video(archive_id, dest_dir, prefer_height=480):
    """Download video from archive.org using yt-dlp."""
    url = f'https://archive.org/details/{archive_id}'
    output = str(dest_dir / f'{archive_id}.%(ext)s')
    cmd = [
        'yt-dlp', url,
        '-f', f'bestvideo[height<={prefer_height}]+bestaudio/best[height<={prefer_height}]/best',
        '--merge-output-format', 'mp4',
        '--no-playlist', '--no-warnings',
        '-o', output, '--quiet',
    ]
    try:
        subprocess.run(cmd, check=True, timeout=300)
        for f in dest_dir.iterdir():
            if f.stem == archive_id and f.suffix in ('.mp4','.mkv','.webm'): return f
    except Exception as e:
        print(f'   ❌ Download failed: {e}')
    return None

def segment_video(video_path, clip_dir, max_dur=10, min_dur=3):
    """Split video into fixed-length clips."""
    total = get_dur(video_path)
    if total <= 0: return []
    clips = []
    t = 0; idx = 0
    while t < total:
        remaining = total - t
        clip_dur = min(max_dur, remaining)
        if clip_dur < min_dur: break
        out = clip_dir / f'archive_{idx:03d}.mp4'
        cmd = ['ffmpeg','-y','-ss',str(t),'-i',str(video_path),'-t',str(clip_dur),
               '-c:v','libx264','-preset','fast','-crf','23','-an',str(out)]
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        if r.returncode == 0 and out.exists():
            clips.append({'path':str(out),'start':round(t,1),'end':round(t+clip_dur,1),'duration':round(clip_dur,1)})
            idx += 1
        t += clip_dur
    return clips

print('✅ Helpers loaded')

In [ ]:
# ============================================================
# DOWNLOAD AND SEGMENT
# ============================================================

all_clips = []

for i, url in enumerate(ARCHIVE_URLS):
    print(f'\n{"="*50}')
    try:
        aid = parse_archive_id(url)
    except ValueError as e:
        print(f'❌ {e}'); continue
    
    print(f'🎬 [{i+1}/{len(ARCHIVE_URLS)}] Archive ID: {aid}')
    print(f'   URL: https://archive.org/details/{aid}')
    
    # Download
    print(f'   Downloading ({PREFER_HEIGHT}p)...', end=' ', flush=True)
    video = download_archive_video(aid, TEMP, PREFER_HEIGHT)
    if not video:
        print('❌ Failed'); continue
    total_dur = get_dur(video)
    print(f'✅ {total_dur:.0f}s')
    
    # Segment
    print(f'   Segmenting into {MIN_CLIP_DURATION}-{MAX_CLIP_DURATION}s clips...')
    clip_dir = ARCHIVE_DIR / aid
    clip_dir.mkdir(exist_ok=True)
    clips = segment_video(video, clip_dir, MAX_CLIP_DURATION, MIN_CLIP_DURATION)
    
    for c in clips: c['archive_id'] = aid; c['source_url'] = f'https://archive.org/details/{aid}'
    all_clips.extend(clips)
    print(f'   ✅ {len(clips)} clips')

# Save index
if all_clips:
    with open(SESSION_DIR / 'archive_clips.json', 'w') as f:
        json.dump({'clips': all_clips, 'total': len(all_clips)}, f, indent=2)
    print(f'\n✅ {len(all_clips)} total clips saved to session')
else:
    print('\n⚠️ No clips generated. Add URLs above and re-run.')

---
## 👀 Preview Clips

In [ ]:
# PREVIEW (show first 6 clips)
if all_clips:
    display(Markdown(f'# 🎬 {len(all_clips)} Archive Clips'))
    for c in all_clips[:6]:
        display(Markdown(f'### {Path(c["path"]).name} ({c["duration"]}s)\n`{c["archive_id"]}` [{c["start"]}-{c["end"]}s]'))
        try: display(Video(c['path'], embed=True, width=480))
        except: print(f'   File: {c["path"]}')
        display(Markdown('---'))
    if len(all_clips) > 6: print(f'... and {len(all_clips)-6} more clips')
else:
    print('No clips to preview.')

---
## 🔀 Use Clips as Found Footage

There are two ways to use these archive clips:

In [ ]:
# OPTION A: Copy selected clips to videos/ folder as scene replacements
# This replaces AI-generated scenes with archive footage

REPLACE_SCENES = {}  # Example: {2: 0, 3: 1} = scene 2 gets clip 0, scene 3 gets clip 1

if REPLACE_SCENES and all_clips:
    for scene_num, clip_idx in REPLACE_SCENES.items():
        if clip_idx < len(all_clips):
            src = Path(all_clips[clip_idx]['path'])
            dst = VIDEOS_DIR / f'scene_{scene_num:03d}.mp4'
            shutil.copy2(src, dst)
            print(f'   ✅ Scene {scene_num} → {src.name} (archive footage)')
    print(f'\nThese will replace AI-generated clips when you run 06_Assemble.')
else:
    print('ℹ️ Set REPLACE_SCENES above to swap AI clips with archive footage.')
    print('   Example: REPLACE_SCENES = {2: 0, 3: 1}')

In [ ]:
# OPTION B: Add all clips as extra footage for the mixer
# The Assembly notebook can pick from these

ADD_ALL_AS_FOOTAGE = False  # Set True to copy all clips to videos/

if ADD_ALL_AS_FOOTAGE and all_clips:
    for i, c in enumerate(all_clips):
        src = Path(c['path'])
        dst = VIDEOS_DIR / f'found_{i:03d}.mp4'
        shutil.copy2(src, dst)
    print(f'✅ Copied {len(all_clips)} clips to videos/ as found footage')
else:
    print('ℹ️ Set ADD_ALL_AS_FOOTAGE = True to use all clips.')

---
## 📊 Clip Index

The full clip index is saved as `archive_clips.json` in your session.

In [ ]:
# SUMMARY
if all_clips:
    total_dur = sum(c['duration'] for c in all_clips)
    sources = set(c['archive_id'] for c in all_clips)
    print(f'📊 Archive Summary:')
    print(f'   Sources: {len(sources)}')
    print(f'   Clips: {len(all_clips)}')
    print(f'   Total footage: {total_dur:.0f}s ({total_dur/60:.1f}min)')
    print(f'   Saved to: {SESSION_DIR / "archive_clips.json"}')
    print(f'\n💡 DISCUSSION:')
    print(f'   This footage is from the public domain Internet Archive.')
    print(f'   What changes when you mix it with AI-generated video?')
    print(f'   Can you tell which scenes are "real" and which are generated?')

---
Next: **02_Script_Writer** (if not done), then **05_Generate** and **06_Assemble**.

---
*ROTBOTS Workshop — LI-MA 2026*